<a href="https://colab.research.google.com/github/AlexandreSantosRodrigues/do-poco-a-acao/blob/main/01_coleta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
from pathlib import Path
import datetime as dt, os

drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/do-poco-a-acao")
for p in ["data/raw/anp", "data/processed", "src", "sql", "reports", "notebooks"]:
    (BASE / p).mkdir(parents=True, exist_ok=True)
os.chdir(BASE)

HOJE = dt.date.today().isoformat()
print("Pasta de trabalho:", Path.cwd(), "| Data da coleta:", HOJE)

Mounted at /content/drive
Pasta de trabalho: /content/drive/MyDrive/do-poco-a-acao | Data da coleta: 2026-10-07


In [2]:
!pip -q install yfinance

In [3]:
from google.colab import files
up = files.upload()
for nome, conteudo in up.items():
    (BASE / "data/raw/anp" / nome).write_bytes(conteudo)   # nome original, sem editar nada
    print("salvo:", nome, f"{len(conteudo)/1e6:.1f} MB")

Saving 2023_08_producao_Presal.xlsx to 2023_08_producao_Presal.xlsx
Saving 2023_08_producao_Terra.xlsx to 2023_08_producao_Terra.xlsx
Saving 2023_07_producao_Terra.xlsx to 2023_07_producao_Terra.xlsx
Saving 2023_08_producao_Mar.xlsx to 2023_08_producao_Mar.xlsx
Saving 2023_06_producao_Terra.xlsx to 2023_06_producao_Terra.xlsx
Saving 2023_07_producao_Mar.xlsx to 2023_07_producao_Mar.xlsx
Saving 2023_07_producao_Presal.xlsx to 2023_07_producao_Presal.xlsx
Saving 2023_06_producao_Presal.xlsx to 2023_06_producao_Presal.xlsx
Saving 2023_05_producao_Presal.xlsx to 2023_05_producao_Presal.xlsx
Saving 2023_05_producao_Terra.xlsx to 2023_05_producao_Terra.xlsx
Saving 2023_06_producao_Mar.xlsx to 2023_06_producao_Mar.xlsx
Saving 2023_04_producao_Presal.xlsx to 2023_04_producao_Presal.xlsx
Saving 2023_04_producao_Terra.xlsx to 2023_04_producao_Terra.xlsx
Saving 2023_05_producao_Mar.xlsx to 2023_05_producao_Mar.xlsx
Saving 2023_03_producao_Presal.xlsx to 2023_03_producao_Presal.xlsx
Saving 2023_03

In [4]:
import requests
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILBRENTEU"
r = requests.get(url, timeout=60); r.raise_for_status()
(BASE / f"data/raw/fred_brent_DCOILBRENTEU_{HOJE}.csv").write_bytes(r.content)
print(r.text[:300])

observation_date,DCOILBRENTEU
1987-05-20,18.63
1987-05-21,18.45
1987-05-22,18.55
1987-05-25,18.60
1987-05-26,18.63
1987-05-27,18.60
1987-05-28,18.60
1987-05-29,18.58
1987-06-01,18.65
1987-06-02,18.68
1987-06-03,18.75
1987-06-04,18.78
1987-06-05,18.65
1987-06-08,18.75
1987-06-09,18.78
1987-06-10,18.7


In [6]:
import json, time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

SERIES = {"ptax_venda": 1, "selic_meta": 432}   # confira os códigos no site do SGS/BCB

def baixa_sgs(codigo, ano_ini=2018, ano_fim=dt.date.today().year, passo=5):
    dados = []

    # Configurando uma sessão com tentativas automáticas (Retries) para lidar com erros 502/503/504
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1,  # Espera 1s, 2s, 4s, 8s, 16s entre as tentativas
        status_forcelist=[500, 502, 503, 504]
    )
    session.mount("https://", HTTPAdapter(max_retries=retries))

    for a in range(ano_ini, ano_fim + 1, passo):
        fim = min(a + passo - 1, ano_fim)
        u = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
             f"?formato=json&dataInicial=01/01/{a}&dataFinal=31/12/{fim}")

        resp = session.get(u, timeout=60)
        resp.raise_for_status()
        dados += resp.json()
        time.sleep(1)
    return dados

for nome, cod in SERIES.items():
    try:
        d = baixa_sgs(cod)
        (BASE / f"data/raw/bcb_{nome}_sgs{cod}_{HOJE}.json").write_text(json.dumps(d), encoding="utf-8")
        print(nome, len(d), "registros")
    except Exception as e:
        print(f"Erro ao baixar a série {nome} (código {cod}): {e}")

ptax_venda 2202 registros
selic_meta 3230 registros


In [7]:
import yfinance as yf, pandas as pd
for t in ["PETR4.SA", "PRIO3.SA"]:
    df = yf.download(t, start="2018-01-01", auto_adjust=False, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)   # só estrutura, sem mexer nos valores
    df.to_csv(BASE / f"data/raw/yf_{t.replace('.SA','')}_{HOJE}.csv")
    print(t, df.shape)

PETR4.SA (2180, 6)
PRIO3.SA (2180, 6)


In [8]:
linhas = ["# Inventário dos dados brutos", f"Coleta em {HOJE}", ""]
for f in sorted((BASE / "data/raw").rglob("*")):
    if f.is_file():
        linhas.append(f"- `{f.relative_to(BASE/'data/raw')}`: {f.stat().st_size/1e6:.2f} MB")
(BASE / "data/raw/INVENTARIO.md").write_text("\n".join(linhas), encoding="utf-8")
print("\n".join(linhas))

# Inventário dos dados brutos
Coleta em 2026-10-07

- `anp/2018_producao_mar.xlsx`: 4.33 MB
- `anp/2018_producao_presal.xlsx`: 0.34 MB
- `anp/2018_producao_terra.xlsx`: 32.11 MB
- `anp/2019_01_producao_mar.xlsx`: 0.40 MB
- `anp/2019_01_producao_presal.xlsx`: 0.05 MB
- `anp/2019_01_producao_terra.xlsx`: 2.84 MB
- `anp/2019_02_producao_mar.xlsx`: 0.39 MB
- `anp/2019_02_producao_presal.xlsx`: 0.09 MB
- `anp/2019_02_producao_terra.xlsx`: 2.98 MB
- `anp/2019_03_producao_mar.xlsx`: 0.39 MB
- `anp/2019_03_producao_presal.xlsx`: 0.06 MB
- `anp/2019_03_producao_terra.xlsx`: 2.78 MB
- `anp/2019_04_producao_mar.xlsx`: 0.41 MB
- `anp/2019_04_producao_presal.xlsx`: 0.07 MB
- `anp/2019_04_producao_terra.xlsx`: 2.82 MB
- `anp/2019_05_producao_mar.xlsx`: 0.41 MB
- `anp/2019_05_producao_presal.xlsx`: 0.07 MB
- `anp/2019_05_producao_terra.xlsx`: 2.85 MB
- `anp/2019_06_producao_mar.xlsx`: 0.38 MB
- `anp/2019_06_producao_presal.xlsx`: 0.07 MB
- `anp/2019_06_producao_terra.xlsx`: 3.09 MB
- `anp/2019_07_pro